<a href="https://colab.research.google.com/github/steveonw/Piper-VITS-TTS-on-Hailo-10H/blob/main/hailo_tts_v14_1_compile_flow_and_decoder_hef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amy → Hailo-10H v14.1 — compile BOTH Flow and Decoder to HEF

v14 established two important facts:

## Flow

All four VITS channel-reversal `Slice(step=-1, axis=1)` nodes were replaced by fixed 1×1 Conv2D permutation matrices.

- h-major equivalence: **max_abs = 0.0**
- w-major equivalence: **max_abs = 0.0**
- h-major DFC parse: **PASS → HAR**
- w-major DFC parse: **PASS → HAR**

So the complete T=148 flow is now parser-compatible.

## Decoder

The complete T=148 decoder already parsed to HAR in v13.1b.

v14's HEF attempt did **not reach optimization**. The harness incorrectly expected Hailo's input-layer shape to be rank 3. Hailo correctly reports shapes including a dynamic batch dimension:

- h-major: `[-1,148,1,192]`
- w-major: `[-1,1,148,192]`

v14.1 strips the leading batch dimension and creates calibration arrays as:

`[N] + HWC`

This notebook attempts:

- decoder h-major: HAR → optimize → HEF
- decoder w-major: HAR → optimize → HEF
- flow h-major: HAR → optimize → HEF
- flow w-major: HAR → optimize → HEF

Calibration is still synthetic for this **compiler/allocation smoke test**. Real Amy latent calibration comes after we know HEF compilation works.


## 1. Locate/upload v14 results and DFC wheel


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, re, time

CONTENT = Path("/content")
RESULTS = CONTENT / "amy_v14_1_results"
RESULTS.mkdir(exist_ok=True)

REQUIRED = {
    "amy_v14_flow_hmajor_flipconv.har",
    "amy_v14_flow_wmajor_flipconv.har",
    "amy_v13_1b_decoder_t148_hmajor.har",
    "amy_v13_1b_decoder_t148_wmajor.har",
}

def names_in_zip(p):
    try:
        with zipfile.ZipFile(p) as z:
            return {Path(x).name for x in z.namelist()}
    except Exception:
        return set()

def locate():
    pack = None
    for p in sorted(CONTENT.glob("*.zip")):
        if REQUIRED.issubset(names_in_zip(p)):
            pack = p
            break
    wheels = sorted(CONTENT.glob("hailo_dataflow_compiler-*.whl"))
    return pack, (wheels[0] if wheels else None)

PACK, WHEEL = locate()
VENV = CONTENT / "hailo-venv"
PY = VENV / "bin" / "python"

if PACK is None or (not PY.exists() and WHEEL is None):
    from google.colab import files
    print("Upload:")
    print(" • amy_hailo_v14_flow_flip_decoder_hef_results.zip")
    if not PY.exists():
        print(" • hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl")
    files.upload()
    PACK, WHEEL = locate()

if PACK is None:
    raise FileNotFoundError("v14 results ZIP not found")
if not PY.exists() and WHEEL is None:
    raise FileNotFoundError("DFC wheel required because /content/hailo-venv is absent")

print("v14 pack:", PACK)
print("Existing Hailo venv:", PY.exists())
print("DFC wheel:", WHEEL)


Upload:
 • amy_hailo_v14_flow_flip_decoder_hef_results.zip
 • hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl


Saving amy_hailo_v14_flow_flip_decoder_hef_results.zip to amy_hailo_v14_flow_flip_decoder_hef_results.zip
Saving hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl to hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl
v14 pack: /content/amy_hailo_v14_flow_flip_decoder_hef_results.zip
Existing Hailo venv: False
DFC wheel: /content/hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl


## 2. Reuse/create isolated Hailo environment


In [ ]:
def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, text=True, **kwargs)

def probe():
    if not PY.exists():
        return None
    return run(
        [str(PY), "-c",
         "import hailo_sdk_client, numpy; "
         "print('Hailo SDK OK'); print('NumPy', numpy.__version__)"],
        capture_output=True
    )

if VENV.exists() and not PY.exists():
    shutil.rmtree(VENV, ignore_errors=True)

if not PY.exists():
    a = run([sys.executable, "-m", "venv", str(VENV)], capture_output=True)
    if a.returncode != 0:
        shutil.rmtree(VENV, ignore_errors=True)
        run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", "virtualenv"], check=True)
        run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)

p = probe()
if p is None or p.returncode != 0:
    run([str(PY), "-m", "pip", "install", "--no-cache-dir", "--upgrade",
         "pip", "setuptools", "wheel"], check=True)
    run([str(PY), "-m", "pip", "install", "--no-cache-dir", str(WHEEL)], check=True)
    p = probe()

print(p.stdout if p else "")
if p is None or p.returncode != 0:
    if p:
        print(p.stderr)
    raise RuntimeError("Hailo DFC environment failed")

print("READY.")


+ /usr/bin/python3 -m venv /content/hailo-venv
+ /usr/bin/python3 -m pip install --no-cache-dir -q virtualenv
+ /usr/bin/python3 -m virtualenv /content/hailo-venv
+ /content/hailo-venv/bin/python -c import hailo_sdk_client, numpy; print('Hailo SDK OK'); print('NumPy', numpy.__version__)
+ /content/hailo-venv/bin/python -m pip install --no-cache-dir --upgrade pip setuptools wheel
+ /content/hailo-venv/bin/python -m pip install --no-cache-dir /content/hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl
+ /content/hailo-venv/bin/python -c import hailo_sdk_client, numpy; print('Hailo SDK OK'); print('NumPy', numpy.__version__)
Hailo SDK OK
NumPy 1.26.4

READY.


## 3. Extract all four successful HARs


In [ ]:
SRC = RESULTS / "source"
SRC.mkdir(exist_ok=True)

with zipfile.ZipFile(PACK) as z:
    z.extractall(SRC)

def find(name):
    hits = list(SRC.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

HARS = {
    "decoder_t148_hmajor": find("amy_v13_1b_decoder_t148_hmajor.har"),
    "decoder_t148_wmajor": find("amy_v13_1b_decoder_t148_wmajor.har"),
    "flow_t148_hmajor": find("amy_v14_flow_hmajor_flipconv.har"),
    "flow_t148_wmajor": find("amy_v14_flow_wmajor_flipconv.har"),
}

for k,v in HARS.items():
    print(k, v, f"{v.stat().st_size/1024/1024:.2f} MiB")


decoder_t148_hmajor /content/amy_v14_1_results/source/amy_v14_results/source/amy_v13_1b_results/amy_v13_1b_decoder_t148_hmajor.har 6.53 MiB
decoder_t148_wmajor /content/amy_v14_1_results/source/amy_v14_results/source/amy_v13_1b_results/amy_v13_1b_decoder_t148_wmajor.har 6.53 MiB
flow_t148_hmajor /content/amy_v14_1_results/source/amy_v14_results/amy_v14_flow_hmajor_flipconv.har 28.19 MiB
flow_t148_wmajor /content/amy_v14_1_results/source/amy_v14_results/amy_v14_flow_wmajor_flipconv.har 28.19 MiB


## 4. Optimize + compile helper

Important Hailo calibration convention:

`input_layer.output_shape` includes batch:

`[-1,H,W,C]`

Calibration arrays therefore use:

`[N,H,W,C]`

For multi-input models, Hailo's own Whisper conversion uses a dictionary keyed by the exact Hailo input-layer names. We do the same for Flow.


In [ ]:
compile_source = r"""
import sys, json, time, traceback
from pathlib import Path
import numpy as np
from hailo_sdk_client import ClientRunner

har_path = Path(sys.argv[1])
outdir = Path(sys.argv[2])
label = sys.argv[3]
kind = sys.argv[4]  # decoder | flow
outdir.mkdir(exist_ok=True, parents=True)

report = {
    "label": label,
    "kind": kind,
    "har": str(har_path),
    "optimize_ok": False,
    "compile_ok": False,
}

def sample_dims(output_shape):
    s = [int(x) for x in output_shape]
    if len(s) == 4 and s[0] in (-1, 1):
        return s[1:]
    if len(s) == 3:
        return s
    raise RuntimeError(f"Unexpected Hailo input-layer shape: {s}")

try:
    runner = ClientRunner(hw_arch="hailo10h", har=str(har_path))
    layers = runner.get_hn_model().get_input_layers()

    layer_info = []
    for layer in layers:
        layer_info.append({
            "name": layer.name,
            "scope": getattr(layer, "scope", None),
            "output_shape": [int(x) for x in layer.output_shape],
            "sample_dims": sample_dims(layer.output_shape),
        })
    report["input_layers"] = layer_info

    rng = np.random.default_rng(14101)
    N = 128

    if kind == "decoder":
        if len(layers) != 1:
            raise RuntimeError(f"Decoder expected one input; got {len(layers)}")

        layer = layers[0]
        dims = sample_dims(layer.output_shape)

        calib = np.empty([N] + dims, dtype=np.float32)
        for i in range(N):
            sigma = [0.5, 0.8, 1.0, 1.5, 2.0][i % 5]
            calib[i] = (rng.standard_normal(dims) * sigma).astype(np.float32)
        np.clip(calib, -6.0, 6.0, out=calib)

        report["calibration"] = {
            "type": "single-array",
            "shape": list(calib.shape),
            "min": float(calib.min()),
            "max": float(calib.max()),
            "mean": float(calib.mean()),
            "std": float(calib.std()),
        }
        np.save(outdir / f"{label}_synthetic_calibration.npy", calib)
        calib_data = calib

    elif kind == "flow":
        if len(layers) != 2:
            raise RuntimeError(f"Flow expected two inputs; got {len(layers)}")

        calib_data = {}
        cal_report = {}

        # Identify activation vs mask from channel count.
        for layer in layers:
            dims = sample_dims(layer.output_shape)
            H, W, C = dims

            if C == 192:
                arr = np.empty([N] + dims, dtype=np.float32)
                for i in range(N):
                    sigma = [0.5, 0.8, 1.0, 1.5, 2.0][i % 5]
                    arr[i] = (rng.standard_normal(dims) * sigma).astype(np.float32)
                np.clip(arr, -6.0, 6.0, out=arr)

            elif C == 1:
                arr = np.zeros([N] + dims, dtype=np.float32)

                # Time is whichever spatial dimension is >1.
                time_axis = 0 if H > 1 else 1
                T = H if H > 1 else W

                for i in range(N):
                    valid_choices = [32, 64, 96, 128, T]
                    valid = min(valid_choices[i % len(valid_choices)], T)

                    if time_axis == 0:
                        arr[i, :valid, :, :] = 1.0
                    else:
                        arr[i, :, :valid, :] = 1.0
            else:
                raise RuntimeError(
                    f"Could not classify Flow input {layer.name} with dims {dims}"
                )

            calib_data[layer.name] = arr
            cal_report[layer.name] = {
                "shape": list(arr.shape),
                "min": float(arr.min()),
                "max": float(arr.max()),
                "mean": float(arr.mean()),
                "std": float(arr.std()),
            }

        report["calibration"] = {
            "type": "dict",
            "keys": list(calib_data.keys()),
            "arrays": cal_report,
        }
        np.savez(outdir / f"{label}_synthetic_calibration.npz", **calib_data)

    else:
        raise RuntimeError(f"Unknown kind {kind}")

    print("INPUT LAYERS")
    print(json.dumps(layer_info, indent=2), flush=True)
    print("CALIBRATION")
    print(json.dumps(report["calibration"], indent=2), flush=True)

    t0 = time.time()
    runner.optimize(calib_data=calib_data, work_dir=str(outdir))
    report["optimize_seconds"] = time.time() - t0
    report["optimize_ok"] = True

    opt_har = outdir / f"{label}_optimized.har"
    runner.save_har(str(opt_har))
    report["optimized_har"] = str(opt_har)
    report["optimized_har_bytes"] = opt_har.stat().st_size

    print("OPTIMIZE_SUCCESS", opt_har, flush=True)

    t0 = time.time()
    hef_bytes = runner.compile()
    report["compile_seconds"] = time.time() - t0
    report["compile_ok"] = True

    hef = outdir / f"{label}.hef"
    hef.write_bytes(hef_bytes)
    report["hef"] = str(hef)
    report["hef_bytes"] = hef.stat().st_size

    compiled_har = outdir / f"{label}_compiled.har"
    runner.save_har(str(compiled_har))
    report["compiled_har"] = str(compiled_har)
    report["compiled_har_bytes"] = compiled_har.stat().st_size

    print("HEF_SUCCESS", hef, hef.stat().st_size, flush=True)

except Exception as e:
    report["exception_type"] = type(e).__name__
    report["exception"] = str(e)
    report["traceback"] = traceback.format_exc()
    print(report["traceback"], flush=True)

finally:
    rp = outdir / f"{label}_compile_report.json"
    rp.write_text(json.dumps(report, indent=2))
    print("REPORT", rp, flush=True)

sys.exit(0 if report["compile_ok"] else 7)
"""

COMPILE = CONTENT / "compile_amy_v14_1.py"
COMPILE.write_text(compile_source)


5490

## 5. Run all four compile attempts — no early break


In [ ]:
def attempt(label, har, kind):
    so = RESULTS / f"{label}_stdout.txt"
    se = RESULTS / f"{label}_stderr.txt"

    with open(so, "w") as a, open(se, "w") as b:
        p = subprocess.run(
            [str(PY), str(COMPILE), str(har), str(RESULTS), label, kind],
            stdout=a, stderr=b, text=True
        )

    stdout = so.read_text(errors="replace")
    stderr = se.read_text(errors="replace")

    print(f"\n{'='*72}")
    print(label)
    print('='*72)
    print(stdout[-30000:])
    if stderr.strip():
        print("----- STDERR -----")
        print(stderr[-20000:])

    rp = RESULTS / f"{label}_compile_report.json"
    report = json.loads(rp.read_text()) if rp.exists() else {
        "compile_ok": False,
        "missing_report": True,
    }

    state = (
        "HEF ✅" if report.get("compile_ok")
        else "OPTIMIZED HAR ✅ / compile failed"
        if report.get("optimize_ok")
        else "optimization not completed"
    )
    print("STATE:", state)
    return report

reports = {}

for label, kind in [
    ("decoder_t148_hmajor", "decoder"),
    ("decoder_t148_wmajor", "decoder"),
    ("flow_t148_hmajor", "flow"),
    ("flow_t148_wmajor", "flow"),
]:
    reports[label] = attempt(label, HARS[label], kind)

(RESULTS / "compile_results.json").write_text(
    json.dumps(reports, indent=2)
)



decoder_t148_hmajor
shed after 28 iterations, Time it took: 5m 7s 905ms
Top common errors:
#1 (20/20) Status: 141, Most common message: Automri finished with too many resources on context_0
[info] Applying selected partition to 2 contexts...
[info] amy_v13_1b_decoder_t148_hmajor Successful Multi Context Partition (duration: 5m 31s)
[info] Validating layers feasibility
[info] deconv1_defuse_conv_defuse_width_feature_reshape: Pass
[info] input_layer1: Pass
[info] deconv1_defuse_d2s_defuse_reshape_f_to_hxw_transposed: Pass
[info] deconv1_defuse_conv_defuse_reshape_hxf_to_w_transposed: Pass
[info] deconv1_defuse_d2s_defuse_width_feature_reshape: Pass
[info] activation1: Pass
[info] activation2: Pass
[info] conv2: Pass
[info] conv3_defuse_width_feature_reshape: Pass
[info] conv1: Pass
[info] conv3_defuse_reshape_hxf_to_w_transposed: Pass
[info] conv4_defuse_reshape_hxf_to_w_transposed: Pass
[info] conv4_defuse_width_feature_reshape: Pass
[info] activation3: Pass
[info] deconv1_defuse_conv_

2816405

## 6. Summarize compiler/allocation barriers


In [ ]:
summary = {}

for label, r in reports.items():
    summary[label] = {
        "kind": r.get("kind"),
        "optimize_ok": r.get("optimize_ok", False),
        "compile_ok": r.get("compile_ok", False),
        "optimize_seconds": r.get("optimize_seconds"),
        "compile_seconds": r.get("compile_seconds"),
        "hef": r.get("hef"),
        "hef_bytes": r.get("hef_bytes"),
        "exception_type": r.get("exception_type"),
        "exception": r.get("exception"),
        "input_layers": r.get("input_layers"),
    }

(RESULTS / "v14_1_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print(json.dumps(summary, indent=2))

decoder_hef = any(
    r.get("compile_ok", False)
    for k,r in reports.items()
    if k.startswith("decoder")
)
flow_hef = any(
    r.get("compile_ok", False)
    for k,r in reports.items()
    if k.startswith("flow")
)

print("\n===== VERDICT =====")
if decoder_hef and flow_hef:
    verdict = "BIG GREEN: at least one Decoder HEF AND one Flow HEF produced."
elif decoder_hef:
    verdict = "GREEN: Decoder HEF produced; Flow's exact optimization/compiler barrier captured."
elif flow_hef:
    verdict = "GREEN: Flow HEF produced; Decoder's exact optimization/compiler barrier captured."
else:
    any_opt = any(r.get("optimize_ok", False) for r in reports.values())
    if any_opt:
        verdict = "YELLOW/GREEN: optimization succeeded for at least one graph; compiler/allocation barrier captured."
    else:
        verdict = "YELLOW: exact optimization barriers captured; no HEF yet."

print(verdict)
(RESULTS / "verdict.txt").write_text(verdict + "\n")


{
  "decoder_t148_hmajor": {
    "kind": "decoder",
    "optimize_ok": true,
    "compile_ok": true,
    "optimize_seconds": 45.01636362075806,
    "compile_seconds": 628.3032650947571,
    "hef": "/content/amy_v14_1_results/decoder_t148_hmajor.hef",
    "hef_bytes": 2318336,
    "exception_type": null,
    "exception": null,
    "input_layers": [
      {
        "name": "amy_v13_1b_decoder_t148_hmajor/input_layer1",
        "scope": "amy_v13_1b_decoder_t148_hmajor",
        "output_shape": [
          -1,
          148,
          1,
          192
        ],
        "sample_dims": [
          148,
          1,
          192
        ]
      }
    ]
  },
  "decoder_t148_wmajor": {
    "kind": "decoder",
    "optimize_ok": true,
    "compile_ok": false,
    "optimize_seconds": 48.84716439247131,
    "compile_seconds": null,
    "hef": null,
    "hef_bytes": null,
    "exception_type": "BackendAllocatorException",
    "exception": "Compilation failed: No successful assignments: deconv2_def

63

## 7. Package results


In [ ]:
from google.colab import files

zip_path = CONTENT / "amy_hailo_v14_1_compile_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(CONTENT)))

print("Created:", zip_path)
print("Send this ZIP back to me.")
files.download(str(zip_path))


Created: /content/amy_hailo_v14_1_compile_results.zip
Send this ZIP back to me.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>